# Notebook 11 — Multi-Layer Perceptron

Algoritmo: Multi-Layer Perceptron. Escalado: sí. n_iter = 20, CV=5, scoring='roc_auc'.

Observaciones: Red neuronal. Necesita escalado. Early stopping.

## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

RAIZ = Path('..').resolve()
sys.path.insert(0, str(RAIZ))

from src.preprocesamiento import preparar, RANDOM_STATE
from src.entrenamiento  import entrenar_y_evaluar

DATA = RAIZ / 'data'

from sklearn.neural_network import MLPClassifier
from scipy.stats import loguniform

## 2. Cargar y preparar datos

In [2]:
datos = preparar(DATA, aplicar_escalado=True)
print(f'Train: {datos.n_train} | Test: {datos.n_test} | Features: {len(datos.cols_features)}')
print(f'scale_pos_weight atraso: {datos.scale_pos_weight["tuvo_atraso"]:.4f}')


Train: 38664 | Test: 9667 | Features: 152
scale_pos_weight atraso: 0.6534


## 3. Clasificador + grid

In [3]:
clf = MLPClassifier(max_iter=300, early_stopping=True, random_state=RANDOM_STATE)

param_distributions = {
        'hidden_layer_sizes': [(100,), (50, 50), (100, 50), (100, 100), (150, 75)],
        'alpha': loguniform(1e-5, 1e-2),
        'learning_rate_init': loguniform(1e-4, 1e-2),
        'activation': ['relu', 'tanh'],
    }

## 4. RandomizedSearchCV (20 iters × CV 5)

In [4]:
resultado = entrenar_y_evaluar(
    nombre='mlp',
    estimador=clf,
    param_distributions=param_distributions,
    datos=datos,
    data_dir=DATA,
    n_iter=20,
    target='tuvo_atraso',
)
resultado


=== MLP ===
CV AUC:          0.87
Test AUC:        0.863
Test F1:         0.8263
Test Precision:  0.8147
Test Recall:     0.8382
Test Accuracy:   0.7868
Tiempo search:   86.23 s
Best params:     {'activation': 'relu', 'alpha': 0.003363987115958792, 'hidden_layer_sizes': (100, 100), 'learning_rate_init': 0.007568292060167619}


{'nombre': 'mlp',
 'target': 'tuvo_atraso',
 'n_iter': 20,
 'cv_folds': 5,
 'tiempo_search_seg': 86.23,
 'best_params': {'activation': 'relu',
  'alpha': 0.003363987115958792,
  'hidden_layer_sizes': (100, 100),
  'learning_rate_init': 0.007568292060167619},
 'best_cv_auc': 0.87,
 'AUC': 0.863,
 'F1': 0.8263,
 'Precision': 0.8147,
 'Recall': 0.8382,
 'Accuracy': 0.7868}